# Avaliação por Partidas Reais — CNN vs Minimax

Este notebook foi isolado para utilizar o ambiente `.venv_tf` (TensorFlow 2.21+), que possui compatibilidade com o modelo `.tflite` exportado no Google Colab (opcodes mais recentes).

**Instrução Importante:** Certifique-se de selecionar o Kernel `.venv_tf` (ou Python do `.venv_tf`) no VS Code antes de rodar a célula abaixo.

In [ ]:
import sys, os
from datetime import datetime
from pathlib import Path

sys.path.insert(0, os.path.abspath('../..'))

from tqdm.auto import tqdm

from gerador_dados.jogo_pontinhos.avaliador_partidas_pontinhos import (
    avaliar_paralelo, imprimir_relatorio,
    todos_labels_canonicos
)

CAMINHO_MODELO  = '../../modelos/pontinhos_pequeno_profundidade_9.tflite'
TAMANHO         = 'pequeno'
N_PARTIDAS      = 200    # total por profundidade (100 CNN 1ª a jogar + 100 CNN 2ª a jogar)
PROFUNDIDADES   = [1, 3, 5, 6]
#PROFUNDIDADES   = [1, 3]
TIMER_MS        = 0      # 0 = sem limite | ex: 5 = Minimax limitado a 5ms/jogada
MAX_WORKERS     = 14

# --- Visualização das "caixas perdidas" (CNN deixou de fechar caixa pronta) ----
SALVAR_CAIXAS_PERDIDAS = True
BASE_VIS = Path('../../visualizacoes/avaliacao_partidas')
# Identifica esta execução (modelo + timestamp), p/ não sobrescrever runs anteriores.
EXEC_ID  = f"{Path(CAMINHO_MODELO).stem}__{datetime.now().strftime('%Y%m%d_%H%M%S')}"
EXEC_DIR = BASE_VIS / EXEC_ID

labels = todos_labels_canonicos(4, 3)

stats_list = []
if __name__ == '__main__':
    if TIMER_MS > 0:
        print(f'Timer ativo: Minimax limitado a {TIMER_MS}ms por jogada')
    if SALVAR_CAIXAS_PERDIDAS:
        print(f'Salvando visualizações de caixas perdidas em: {EXEC_DIR}')

    for prof in PROFUNDIDADES:
        nome = f'Minimax(p<={prof}, {TIMER_MS}ms)' if TIMER_MS > 0 else f'Minimax(p={prof})'

        with tqdm(total=N_PARTIDAS, desc=f'{nome:25s}', unit='partida', leave=True) as pbar:
            # [vitórias, empates, derrotas, opp_perdidas, opp_total]
            c = [0, 0, 0, 0, 0]

            def _cb(completed, total, result, _pbar=pbar, _c=c):
                if result['vencedor'] == 1:   _c[0] += 1
                elif result['vencedor'] == 0:  _c[1] += 1
                else:                          _c[2] += 1
                _c[3] += result.get('opp_perdidas_a1', 0)
                _c[4] += result.get('opp_total_a1', 0)
                _pbar.update(completed - _pbar.n)
                postfix = {'V': _c[0], 'E': _c[1], 'D': _c[2]}
                if _c[4] > 0:
                    postfix['?caixa'] = f"{_c[3]}/{_c[4]}"
                _pbar.set_postfix(postfix)

            saida_misses = (EXEC_DIR / f'minimax_p{prof}') if SALVAR_CAIXAS_PERDIDAS else None

            s = avaliar_paralelo(
                CAMINHO_MODELO, labels, prof, nome, TAMANHO,
                N_PARTIDAS, TIMER_MS, MAX_WORKERS,
                progress_callback=_cb,
                salvar_caixas_perdidas_em=saida_misses,
            )
        stats_list.append(s)

    imprimir_relatorio(stats_list)

d:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\.venv_tf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Salvando visualizações de caixas perdidas em: ..\..\visualizacoes\avaliacao_partidas\pontinhos_pequeno_profundidade_9__20260508_145334


Minimax(p=1)             :   0%|          | 0/200 [00:00<?, ?partida/s]

## Resultado Profundidade 6

In [ ]:
# Avaliando contra Minimax(p=1) em paralelo (multi-core)...
# Avaliando contra Minimax(p=3) em paralelo (multi-core)...
# Avaliando contra Minimax(p=5) em paralelo (multi-core)...
# Avaliando contra Minimax(p=6) em paralelo (multi-core)...
# 
# ========================================================================
# AVALIAÇÃO POR PARTIDAS REAIS — CNN vs Minimax
# **CNN TREINADA COM TABULEIROS GERADOS DE FORMA ALEATÓRIA**
# ========================================================================
# 
#   Adversário: Minimax(p=1)  (200 partidas)
#   Vitórias CNN          192  ( 96.0%)
#   Empates                 4  (  2.0%)
#   Derrotas CNN            4  (  2.0%)
#   Tempo médio CNN:  0.06 ms/jogada
#   Tempo médio Minimax(p=1): 0.2 ms/jogada
#   CNN é 3× mais rápida
# 
#   Adversário: Minimax(p=3)  (200 partidas)
#   Vitórias CNN          105  ( 52.5%)
#   Empates                26  ( 13.0%)
#   Derrotas CNN           69  ( 34.5%)
#   Tempo médio CNN:  0.10 ms/jogada
#   Tempo médio Minimax(p=3): 82.4 ms/jogada
#   CNN é 847× mais rápida
# 
#   Adversário: Minimax(p=5)  (200 partidas)
#   Vitórias CNN           44  ( 22.0%)
#   Empates                13  (  6.5%)
#   Derrotas CNN          143  ( 71.5%)
#   Tempo médio CNN:  0.13 ms/jogada
#   Tempo médio Minimax(p=5): 1639.1 ms/jogada
#   CNN é 12519× mais rápida
# 
#   Adversário: Minimax(p=6)  (200 partidas)
#   Vitórias CNN            3  (  1.5%)
#   Empates                10  (  5.0%)
#   Derrotas CNN          187  ( 93.5%)
#   Tempo médio CNN:  0.15 ms/jogada
#   Tempo médio Minimax(p=6): 5107.0 ms/jogada
#   CNN é 34794× mais rápida
# 
# ========================================================================

## Resultado Profundidade 7

In [ ]:
# Avaliando contra Minimax(p=1) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# Avaliando contra Minimax(p=3) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# Avaliando contra Minimax(p=5) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# Avaliando contra Minimax(p=6) em paralelo (multi-core)...
#   Progresso: [200/200] 100.0% concluído...
# 
# ========================================================================
# AVALIAÇÃO POR PARTIDAS REAIS — CNN vs Minimax
# ========================================================================
# 
#   Adversário: Minimax(p=1)  (200 partidas)
#   Vitórias CNN          184  ( 92.0%)
#   Empates                12  (  6.0%)
#   Derrotas CNN            4  (  2.0%)
#   Tempo médio CNN:  0.09 ms/jogada
#   Tempo médio Minimax(p=1): 0.2 ms/jogada
#   CNN é 2× mais rápida
# 
#   Adversário: Minimax(p=3)  (200 partidas)
#   Vitórias CNN          120  ( 60.0%)
#   Empates                54  ( 27.0%)
#   Derrotas CNN           26  ( 13.0%)
#   Tempo médio CNN:  0.11 ms/jogada
#   Tempo médio Minimax(p=3): 80.6 ms/jogada
#   CNN é 711× mais rápida
# 
#   Adversário: Minimax(p=5)  (200 partidas)
#   Vitórias CNN           89  ( 44.5%)
#   Empates                59  ( 29.5%)
#   Derrotas CNN           52  ( 26.0%)
#   Tempo médio CNN:  0.13 ms/jogada
#   Tempo médio Minimax(p=5): 1627.1 ms/jogada
#   CNN é 12419× mais rápida
# 
#   Adversário: Minimax(p=6)  (200 partidas)
#   Vitórias CNN           80  ( 40.0%)
#   Empates                46  ( 23.0%)
#   Derrotas CNN           74  ( 37.0%)
#   Tempo médio CNN:  0.14 ms/jogada
#   Tempo médio Minimax(p=6): 5820.3 ms/jogada
#   CNN é 41782× mais rápida
# 
# ========================================================================